In [11]:
#conda activate TLS
#pip uninstall pymaftools
#pip install --no-cache-dir git+https://github.com/xu62u4u6/pymaftools.git
from pymaftools import *
import sys
sys.path.append("/home/data/data_YM313112004/TLS_model/code/utils/")
cancer_type = "PAAD"
random_seed = 42

In [12]:
# 不用重啟 kernel，Notebook 就會自動載入更新後的函式
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Load Data

In [13]:
from pymaftools import PivotTable
PAAD_rna = PivotTable.read_sqlite(f"/home/data/data_YM313112004/TLS_model/results/{cancer_type}/table/rna_table.sqlite")
PAAD_meth = PivotTable.read_sqlite(f"/home/data/data_YM313112004/TLS_model/results/{cancer_type}/table/meth_table.sqlite")
PAAD_mut = PivotTable.read_sqlite(f"/home/data/data_YM313112004/TLS_model/results/{cancer_type}/table/mutation_table.sqlite")
PAAD_cnv = PivotTable.read_sqlite(f"/home/data/data_YM313112004/TLS_model/results/{cancer_type}/table/cnv_cluster_table_{random_seed}.sqlite")  # 已經 cluster

[PivotTable] loaded from /home/data/data_YM313112004/TLS_model/results/PAAD/table/rna_table.sqlite
[PivotTable] loaded from /home/data/data_YM313112004/TLS_model/results/PAAD/table/meth_table.sqlite
[PivotTable] loaded from /home/data/data_YM313112004/TLS_model/results/PAAD/table/mutation_table.sqlite
[PivotTable] loaded from /home/data/data_YM313112004/TLS_model/results/PAAD/table/cnv_cluster_table_42.sqlite


In [14]:
train_samples = PAAD_rna.sample_metadata[f"split_{random_seed}"] == "train"
test_samples = PAAD_rna.sample_metadata[f"split_{random_seed}"] == "test"
rna_train = PAAD_rna.subset(samples=train_samples)
rna_test = PAAD_rna.subset(samples=test_samples)

train_samples = PAAD_meth.sample_metadata[f"split_{random_seed}"] == "train"
test_samples = PAAD_meth.sample_metadata[f"split_{random_seed}"] == "test"
meth_train = PAAD_meth.subset(samples=train_samples)
meth_test = PAAD_meth.subset(samples=test_samples)

train_samples = PAAD_mut.sample_metadata[f"split_{random_seed}"] == "train"
test_samples = PAAD_mut.sample_metadata[f"split_{random_seed}"] == "test"
mut_train = PAAD_mut.subset(samples=train_samples)
mut_test = PAAD_mut.subset(samples=test_samples)

train_samples = PAAD_cnv.sample_metadata[f"split_{random_seed}"] == "train"
test_samples = PAAD_cnv.sample_metadata[f"split_{random_seed}"] == "test"
cnv_train = PAAD_cnv.subset(samples=train_samples)
cnv_test = PAAD_cnv.subset(samples=test_samples)

### Concat and Tree Importance

In [15]:
from model_utils import concat_omics
PAAD_train = concat_omics([rna_train, meth_train, mut_train, cnv_train], keys=["RNA", "Meth", "Mutation", "CNV"], axis=0)
PAAD_test = concat_omics([rna_test, meth_test, mut_test, cnv_test], keys=["RNA", "Meth", "Mutation", "CNV"], axis=0)
PAAD_train

sample,TCGA-2L-AAQA-01A,TCGA-2L-AAQL-01A,TCGA-2J-AAB4-01A,TCGA-IB-A6UG-01A,TCGA-HZ-A77O-01A,TCGA-3A-A9IL-01A,TCGA-2J-AABO-01A,TCGA-H6-8124-01A,TCGA-IB-A7M4-01A,TCGA-FB-A5VM-01A,...,TCGA-US-A77E-01A,TCGA-IB-A5SQ-01A,TCGA-IB-AAUM-01A,TCGA-FB-AAQ3-01A,TCGA-IB-7887-01A,TCGA-3A-A9IH-01A,TCGA-F2-A7TX-01A,TCGA-HZ-7924-01A,TCGA-IB-7890-01A,TCGA-IB-7654-01A
RNA_A1BG,0.193330,0.108892,0.197111,0.081749,0.000000,1.081749,0.159887,0.074231,0.075601,0.224411,...,0.175557,0.179256,0.037031,0.072586,0.059909,0.116099,0.000000,0.059494,0.278698,0.139862
RNA_A1CF,1.845470,1.654161,1.610228,0.901263,0.743472,3.743623,1.724738,0.750092,0.972913,0.224287,...,1.770491,2.363816,1.077175,1.415975,0.974603,0.734048,1.160146,4.186960,1.398898,1.648373
RNA_A2M,7.074512,8.349059,8.484469,7.083435,7.407824,8.176749,9.015322,7.763216,6.653587,8.083053,...,8.398715,9.593737,7.382747,6.919648,7.820449,8.144368,7.819523,9.625054,9.237116,8.575170
RNA_A2ML1,0.214374,0.106616,0.064193,2.738833,2.330128,0.091124,2.903347,2.855591,2.494953,4.071891,...,1.230818,1.167486,0.149259,0.354678,0.240742,3.088057,1.604783,0.050049,4.204876,0.850479
RNA_A3GALT2,0.247077,0.106482,0.000000,0.424062,0.083656,0.334111,0.297485,0.095992,0.000000,0.000000,...,0.171847,0.453544,0.139600,0.000000,0.288063,0.408929,0.469052,0.058247,0.245009,0.122341
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
CNV_160,0.009000,-0.004000,-0.051000,0.214000,0.046000,-0.587000,0.025750,-0.276000,0.005000,-0.029000,...,0.006000,0.000000,0.876000,-0.050000,0.109000,0.033000,0.139000,0.057000,0.015000,0.073000
CNV_161,0.000000,0.170000,-0.016000,-0.353000,-0.660000,-1.016000,-0.661000,-0.642000,-0.583000,-1.001000,...,-0.023000,-0.448000,-0.647000,-0.496000,-0.266000,0.062000,0.164000,-0.017000,0.026000,-0.386000
CNV_162,0.005000,-0.031000,-0.010000,-0.052000,0.471000,-0.009000,0.099000,-0.027000,-0.368000,-0.292000,...,-0.035000,-0.163000,-0.069000,0.026000,-0.006000,-0.308000,-0.220000,-0.024000,-0.336000,-0.014000
CNV_163,-0.701000,0.202000,-1.293000,-1.293000,-1.266000,0.510000,0.051000,-0.705000,0.011000,0.000000,...,0.377000,0.118000,0.007000,-1.293000,0.021000,-0.652000,0.145000,0.030000,-0.311000,0.000000


In [16]:
top_features_rna = np.load(f"/home/data/data_YM313112004/TLS_model/results/{cancer_type}/selected_features/top_features_rna_mrmr_{random_seed}.npy", allow_pickle=True)
rna_train_top = rna_train.subset(features=top_features_rna)
rna_test_top = rna_test.subset(features=top_features_rna)
top_features_meth = np.load(f"/home/data/data_YM313112004/TLS_model/results/{cancer_type}/selected_features/top_features_meth_mrmr_{random_seed}.npy", allow_pickle=True)
meth_train_top = meth_train.subset(features=top_features_meth)
meth_test_top = meth_test.subset(features=top_features_meth)
top_features_mut = np.load(f"/home/data/data_YM313112004/TLS_model/results/{cancer_type}/selected_features/top_features_mut_mrmr_{random_seed}.npy", allow_pickle=True)
mut_train_top = mut_train.subset(features=top_features_mut)
mut_test_top = mut_test.subset(features=top_features_mut)
top_features_cnv = np.load(f"/home/data/data_YM313112004/TLS_model/results/{cancer_type}/selected_features/top_features_cnv_mrmr_{random_seed}.npy", allow_pickle=True)
cnv_train_top = cnv_train.subset(features=top_features_cnv)
cnv_test_top = cnv_test.subset(features=top_features_cnv)

from model_utils import concat_omics
concat_train = concat_omics([rna_train_top, meth_train_top, mut_train_top, cnv_train_top], keys=["RNA", "Meth", "Mutation", "CNV"], axis=0)
concat_test = concat_omics([rna_test_top, meth_test_top, mut_test_top, cnv_test_top], keys=["RNA", "Meth", "Mutation", "CNV"], axis=0)
concat_train

sample,TCGA-2L-AAQA-01A,TCGA-2L-AAQL-01A,TCGA-2J-AAB4-01A,TCGA-IB-A6UG-01A,TCGA-HZ-A77O-01A,TCGA-3A-A9IL-01A,TCGA-2J-AABO-01A,TCGA-H6-8124-01A,TCGA-IB-A7M4-01A,TCGA-FB-A5VM-01A,...,TCGA-US-A77E-01A,TCGA-IB-A5SQ-01A,TCGA-IB-AAUM-01A,TCGA-FB-AAQ3-01A,TCGA-IB-7887-01A,TCGA-3A-A9IH-01A,TCGA-F2-A7TX-01A,TCGA-HZ-7924-01A,TCGA-IB-7890-01A,TCGA-IB-7654-01A
RNA_CR2,0.037171,0.000000,0.110898,3.730510,0.277985,0.019488,1.625738,0.795019,4.416103,2.378955,...,3.535356,3.973804,2.256558,0.059632,0.096802,2.023362,0.974970,3.438240,1.516519,3.286068
RNA_NOMO2,2.405775,0.951439,2.617651,1.920065,2.780982,2.233949,2.041664,3.851769,2.402995,1.193015,...,1.991789,2.081851,3.257373,0.313478,3.131112,2.575845,2.247077,3.101263,2.796079,3.690518
RNA_H4C5,0.060462,0.000000,0.727964,1.269452,0.113834,0.393636,2.852518,1.868529,2.047120,0.445727,...,0.521051,1.214995,1.530121,0.760093,2.028003,0.420294,0.287236,0.887447,0.653610,0.349252
RNA_HMGCS2,6.043690,3.603775,2.967851,0.212881,2.342498,4.854564,3.813504,2.138716,1.209890,0.681134,...,4.864008,2.245435,4.722406,5.277936,4.096405,1.697640,6.997578,9.188577,0.601126,2.167775
RNA_CCL17,2.566523,3.334969,4.133777,3.006693,3.116265,2.313478,5.415583,0.422341,5.931042,5.061828,...,2.858061,7.211881,3.968275,4.417765,0.789103,4.034357,2.522884,1.535754,2.592684,2.445012
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
CNV_13,-0.127600,0.054200,-0.084733,0.061400,-0.264600,0.460933,-0.035200,-0.093000,-0.101067,-0.398333,...,0.048533,-0.356533,-0.114933,-0.066800,-0.208267,-0.306733,-0.311800,-0.150267,-0.318533,-0.016667
CNV_8,-0.040234,-0.252484,-0.030622,-0.220176,-0.188463,-0.107888,-0.226255,-0.672229,-0.035426,0.001569,...,-0.015207,-0.053457,-0.029191,-0.088404,-0.286426,-0.083085,-0.204000,-0.048793,-0.128564,-0.021000
CNV_116,-0.591000,-0.236000,-0.029000,-0.789000,-1.143000,-1.116000,-0.179000,0.016000,-0.905000,-0.775000,...,0.006000,-0.034000,-0.066000,-0.186000,-0.012000,0.066000,-0.029000,-0.005000,-1.137000,0.012000
CNV_115,-0.136000,-0.049000,-0.297000,-0.192000,0.175000,-0.574000,-0.163000,-0.628000,-0.040000,0.035000,...,0.010000,0.014000,-0.044000,-0.002000,0.035000,-1.070000,-0.025000,0.987000,-0.243000,0.005000


In [17]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, average_precision_score

# ========= 1) Load 3 feature sets =========
feature_paths = {
    "RF_feature_set": f"/home/data/data_YM313112004/TLS_model/results/{cancer_type}/selected_features/top_features_concat_mrmr_rf_42.npy",
    "XGB_feature_set": f"/home/data/data_YM313112004/TLS_model/results/{cancer_type}/selected_features/top_features_concat_mrmr_xgb_42.npy",
    "CatBoost_feature_set": f"/home/data/data_YM313112004/TLS_model/results/{cancer_type}/selected_features/top_features_concat_mrmr_catboost_42.npy",
}

rf_set = set(np.load(feature_paths["RF_feature_set"], allow_pickle=True).tolist())
xgb_set = set(np.load(feature_paths["XGB_feature_set"], allow_pickle=True).tolist())
cat_set = set(np.load(feature_paths["CatBoost_feature_set"], allow_pickle=True).tolist())

# ========= 2) Build C3 groups =========
feature_groups = {
    "C3_1_union": sorted(rf_set | xgb_set | cat_set),  # 至少1個模型選到
    "C3_2_atleast2": sorted((rf_set & xgb_set) | (rf_set & cat_set) | (xgb_set & cat_set)),  # 至少2個
    "C3_3_intersection": sorted(rf_set & xgb_set & cat_set),  # 3個都選到
}
print({k: len(v) for k, v in feature_groups.items()})

# ========= 3) Compare: (C3_1/C3_2/C3_3) x (RF/XGB/CatBoost) =========
all_results = []
detailed_reports = {}

model_list = ["RandomForest", "XGBoost", "CatBoost"] 

for group_name, feats in feature_groups.items():
    for model_name in model_list:
        # 建立基礎模型
        from model_utils import build_base_model
        base_model = build_base_model(model_name, concat_train, random_state=42)
        # 針對 test set 評估模型表現
        from model_utils import evaluate_model
        report_train, report, y_train, y_pred_prob, y_test, y_test_pred_prob, best_cutoff = evaluate_model(
            concat_train,
            concat_test,
            top_features=feats,
            scaler_dict={},
            model=base_model
        )

        y_train_pred = (y_pred_prob >= best_cutoff).astype(int)
        y_test_pred = (y_test_pred_prob >= best_cutoff).astype(int)

        row = {
            "feature_group": group_name,
            "model": model_name,
            "n_features": len(feats),
            "best_cutoff": float(best_cutoff),
            "train_f1": f1_score(y_train, y_train_pred),
            "train_acc": accuracy_score(y_train, y_train_pred),
            "train_auc": roc_auc_score(y_train, y_pred_prob),
            "train_auprc": average_precision_score(y_train, y_pred_prob),
            "test_f1": f1_score(y_test, y_test_pred),
            "test_acc": accuracy_score(y_test, y_test_pred),
            "test_auc": roc_auc_score(y_test, y_test_pred_prob),
            "test_auprc": average_precision_score(y_test, y_test_pred_prob),
        }
        all_results.append(row)
        detailed_reports[(group_name, model_name)] = report

        print(f"\n===== {group_name} | {model_name} =====")
        print(report)

result_df = pd.DataFrame(all_results).sort_values(
    ["test_auc", "test_auprc", "test_f1"], ascending=False
).reset_index(drop=True)

display(result_df)

{'C3_1_union': 45, 'C3_2_atleast2': 24, 'C3_3_intersection': 9}
Selected top 45 features
Best cutoff in AUROC: 0.539944909895946
Train | F1-score: 0.928 | Accuracy: 0.949 | AUC: 0.985 | AUPRC: 0.975
Test  | F1-score: 0.640 | Accuracy: 0.743 | AUC: 0.839 | AUPRC: 0.795

===== C3_1_union | RandomForest =====
              precision    recall  f1-score   support

TLS-negative      0.783     0.818     0.800        22
TLS-positive      0.667     0.615     0.640        13

    accuracy                          0.743        35
   macro avg      0.725     0.717     0.720        35
weighted avg      0.740     0.743     0.741        35

Selected top 45 features
Best cutoff in AUROC: 0.6441149413585663
Train | F1-score: 0.947 | Accuracy: 0.964 | AUC: 0.993 | AUPRC: 0.989
Test  | F1-score: 0.667 | Accuracy: 0.771 | AUC: 0.815 | AUPRC: 0.780

===== C3_1_union | XGBoost =====
              precision    recall  f1-score   support

TLS-negative      0.792     0.864     0.826        22
TLS-positive    

,feature_group,model,n_features,best_cutoff,train_f1,train_acc,train_auc,train_auprc,test_f1,test_acc,test_auc,test_auprc
0,C3_2_atleast2,RandomForest,24,0.509153,0.918367,0.941606,0.977737,0.963700,0.640000,0.742857,0.846154,0.810595
1,C3_1_union,RandomForest,45,0.539945,0.927835,0.948905,0.984926,0.974752,0.640000,0.742857,0.839161,0.794957
2,C3_1_union,CatBoost,45,0.514739,0.950495,0.963504,0.991883,0.985140,0.640000,0.742857,0.832168,0.797625
3,C3_3_intersection,RandomForest,9,0.489932,0.849057,0.883212,0.961039,0.934826,0.692308,0.771429,0.825175,0.757713
4,C3_3_intersection,CatBoost,9,0.496655,0.864865,0.890511,0.954082,0.902468,0.640000,0.742857,0.821678,0.787884
5,C3_2_atleast2,CatBoost,24,0.521408,0.949495,0.963504,0.982607,0.963288,0.615385,0.714286,0.821678,0.765090
6,C3_2_atleast2,XGBoost,24,0.598052,0.927835,0.948905,0.978200,0.957087,0.560000,0.685714,0.818182,0.762562
7,C3_1_union,XGBoost,45,0.644115,0.947368,0.963504,0.993043,0.988522,0.666667,0.771429,0.814685,0.779765
8,C3_3_intersection,XGBoost,9,0.429801,0.857143,0.883212,0.952226,0.893870,0.615385,0.714286,0.790210,0.695633
